# 10 — Hybrid models (R4)

Three ways to combine learned and hand-made knowledge, all built on the Section 6 contract
(every branch → 64-number embedding → fusion → R2's shared regression head):

| ID | Preset | Branches | Prediction |
|----|--------|----------|------------|
| E3 | H1 | LSTM(IMU) + MLP(104 features + 4 physics) | direct |
| E4 | H2 (main) | same as H1 | `a·phys_mag + b` **+ learned residual**, last layer zero-initialised |
| E6 | H3 | LSTM(IMU) + camera CNN | direct |
| E7 | best of E3/E4/E6 **on validation** + 10 optical-flow numbers | | |
| E4vec | H2 with the 4-value head (mag + dx, dy, dz) for drift correction (C1) | | |
| IMUlstm / IMUcnn | branch ablation: raw IMU only (no features, no physics), LSTM or 1D-CNN encoder | | |
| E4cnn | H2 with the 1D-CNN as IMU encoder instead of the LSTM | | |
| `*_min10` | sensitivity: same model, no checkpoint before epoch 10 (warm-up) | | |
| ML_FEAT / ML_PHYS | Lasso on 104 features (/ + 4 physics numbers), same windows, alpha chosen on validation | | |
| PHYS | physics alone, `a·phys_mag + b` (same a, b as H2) | | |

Training: `python scripts/train_hybrid.py` (≈ 15–30 min on CPU, 3 seeds × 2 folds).
This notebook only **reads** the saved results — it never trains and never touches the test flight for selection.

In [ ]:
import os, sys, json
from pathlib import Path
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
FIG = ROOT / "figures" / "dl"; FIG.mkdir(parents=True, exist_ok=True)
FOLDS = ["leave_MH_01_easy", "leave_MH_02_easy"]
runs = pd.read_csv(ROOT / "results/metrics/hybrid_runs.csv")
summary = pd.read_csv(ROOT / "results/metrics/hybrid_summary.csv")
phase1 = pd.read_csv(ROOT / "results/metrics/err_mag_m.csv")

## 1. Fusion diagram

In [ ]:
def box(ax, x, y, w, h, text, color):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02", fc=color, ec="#333", lw=1))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=8.5)
def arrow(ax, x0, y0, x1, y1):
    ax.annotate("", (x1, y1), (x0, y0), arrowprops=dict(arrowstyle="->", lw=1, color="#333"))

fig, ax = plt.subplots(figsize=(10, 4.4)); ax.set_xlim(0, 10); ax.set_ylim(0, 4.6); ax.axis("off")
inputs = [(3.6, "IMU window\n(6 × 400)", "LSTM encoder\n(R2)"),
          (2.3, "104 features\n+ 4 physics [+ 10 flow]", "Feature MLP\n108→64→64"),
          (1.0, "3 frames\n(3 × 64 × 64)", "Camera CNN\n(R3)")]
for y, src, enc in inputs:
    box(ax, 0.1, y, 1.9, 0.75, src, "#e8eef7"); box(ax, 2.5, y, 1.7, 0.75, enc, "#d5e8d4")
    arrow(ax, 2.0, y + .37, 2.5, y + .37); arrow(ax, 4.2, y + .37, 4.75, 2.65)
    ax.text(4.3, y + .55, "64", fontsize=7, color="#555")
box(ax, 4.75, 2.25, 1.5, 0.8, "concat →\nLinear·ReLU·Dropout\n→ 64", "#fff2cc")
box(ax, 6.7, 2.25, 1.3, 0.8, "Regression head\n(R2, shared)\nz", "#fff2cc")
arrow(ax, 6.25, 2.65, 6.7, 2.65)
box(ax, 8.4, 3.25, 1.5, 0.85, "H1 / H3:\nȳ + s·z", "#f8cecc")
box(ax, 8.4, 1.25, 1.5, 0.95, "H2 (main):\na·phys_mag + b\n+ s·z", "#f8cecc")
arrow(ax, 8.0, 2.8, 8.4, 3.6); arrow(ax, 8.0, 2.5, 8.4, 1.75)
ax.text(5.0, 0.35, "a, b, ȳ, s and physics scaling: fitted on the inner-training windows only. "
        "H2's last layer starts at 0 → epoch 0 = physics exactly.", fontsize=8, ha="center")
ax.text(5.0, 4.45, "Branch flags switch IMU / features / camera on or off; a missing camera gives a zero embedding.",
        fontsize=8, ha="center")
fig.tight_layout(); fig.savefig(FIG / "hybrid_architecture.png", dpi=150); plt.show()

## 2. Results table (test flight, mean ± std over 3 seeds)

In [ ]:
lasso = phase1[phase1.model == "lasso"].set_index("fold_id")
mean_base = phase1[phase1.model == "mean"].set_index("fold_id")
e12 = pd.read_csv(ROOT / "results/metrics/imu_e1_e2.csv").set_index(["experiment", "fold_id"])
rows = []
for fold in FOLDS:
    rows.append(dict(model="Mean baseline (Phase 1)", fold=fold, mae_cm=100 * mean_base.loc[fold, "mae"], r2=mean_base.loc[fold, "r2"]))
    rows.append(dict(model="Lasso (Phase 1)", fold=fold, mae_cm=100 * lasso.loc[fold, "mae"], r2=lasso.loc[fold, "r2"]))
    for e in ["E1", "E2"]:
        rows.append(dict(model=f"{e} (R2, seed 42)", fold=fold, mae_cm=100 * e12.loc[(e, fold), "test_mae_m"], r2=e12.loc[(e, fold), "test_r2"]))
    for exp, g in summary[summary.fold_id == fold].groupby("experiment"):
        g = g.iloc[0]
        rows.append(dict(model=exp, fold=fold, mae_cm=100 * g.test_mae_mean, mae_sd=100 * g.test_mae_std,
                         r2=g.test_r2_mean, r2_sd=g.test_r2_std))
table = pd.DataFrame(rows)
wide = table.pivot_table(index="model", columns="fold", values=["mae_cm", "r2"], sort=False)
wide[("mae_cm", "mean")] = wide["mae_cm"].mean(axis=1); wide[("r2", "mean")] = wide["r2"].mean(axis=1)
wide.round(3)

In [ ]:
order = ["PHYS", "IMUcnn", "E3_min10", "E4", "E4cnn", "E6_min10", "E7"]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, fold in zip(axes, FOLDS):
    s = summary[(summary.fold_id == fold)].set_index("experiment").reindex(order)
    colors = ["#999999", "#9673a6", "#6c8ebf", "#d79b00", "#e6b85c", "#82b366", "#b85450"]
    ax.bar(order, 100 * s.test_mae_mean, yerr=100 * s.test_mae_std.fillna(0), color=colors, capsize=4)
    ax.axhline(100 * lasso.loc[fold, "mae"], ls="--", c="k", lw=1, label="Lasso (Phase 1)")
    ax.axhline(100 * mean_base.loc[fold, "mae"], ls=":", c="k", lw=1, label="Mean baseline")
    ax.set_title(f"{fold} (test = {fold[6:]})"); ax.set_ylabel("test MAE (cm)")
axes[0].legend(fontsize=8)
fig.suptitle("R4 models vs physics alone (PHYS); mean ± std over 3 seeds; E3/E6 with 10-epoch warm-up")
fig.tight_layout(); fig.savefig(FIG / "hybrid_comparison.png", dpi=150); plt.show()

## 3. Main hybrid H2 (E4) vs physics alone

The residual head starts at zero, so **epoch 0 is exactly the physics prediction**. Early stopping on the
validation block may keep epoch 0 (the network found nothing that generalises) — that is the designed
safe failure mode, not a bug.

In [ ]:
h2 = runs[runs.experiment == "E4"]
phys = runs[runs.experiment == "PHYS"].set_index("fold_id")
cmp = h2.groupby("fold_id").agg(h2_mae_cm=("test_mae", lambda v: 100 * v.mean()), h2_mae_sd=("test_mae", lambda v: 100 * v.std()),
                                h2_r2=("test_r2", "mean"), h2_r2_sd=("test_r2", "std"),
                                best_epochs=("best_epoch", lambda v: list(v)))
cmp["val_gain_cm"] = (100 * (h2.start_val_mae_m - h2.best_val_mae_m)).groupby(h2.fold_id).mean()
cmp["phys_mae_cm"] = 100 * phys.test_mae; cmp["phys_r2"] = phys.test_r2
cmp["delta_mae_cm"] = cmp.h2_mae_cm - cmp.phys_mae_cm
cmp.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, fold in zip(axes, FOLDS):
    for seed in sorted(h2.seed.unique()):
        name = f"E4_{fold}" if seed == 42 else f"E4_seed{seed}_{fold}"
        c = pd.read_csv(ROOT / f"results/metrics/{name}_loss.csv")
        ax.plot(c.epoch, 100 * c.val_mae, label=f"seed {seed}")
    ax.axhline(100 * c.val_mae.iloc[0], c="k", ls="--", lw=1, label="physics (epoch 0)")
    ax.set_title(f"H2 validation MAE — {fold}"); ax.set_xlabel("epoch"); ax.set_ylabel("val MAE (cm)")
axes[0].legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "hybrid_h2_curves.png", dpi=150); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5.5), sharex=False)
for ax, fold in zip(axes, FOLDS):
    p = pd.read_csv(ROOT / f"results/predictions/PHYS_{fold}.csv")
    q = pd.read_csv(ROOT / f"results/predictions/E4_{fold}.csv")
    t = p.window_id * 0.25
    ax.plot(t, 100 * p.y_true, c="k", lw=1, label="truth")
    ax.plot(t, 100 * p.y_pred, lw=1, label="physics only")
    ax.plot(t, 100 * q.y_pred, lw=1, ls="--", label="H2 (seed 42)")
    ax.set_ylabel("err_mag (cm)"); ax.set_title(f"test flight {fold[6:]}")
axes[-1].set_xlabel("window start (s)"); axes[0].legend(fontsize=8, ncol=3)
fig.tight_layout(); fig.savefig(FIG / "hybrid_h2_over_time.png", dpi=150); plt.show()

### ML reference under the Phase 2 rules

Lasso is fitted on the same inner-training windows and its alpha is chosen on the same validation block that
picks the DL checkpoints. The low-variance landing block pushes alpha up: without physics, Lasso can zero every
feature; with physics, it keeps only `phys_mag` but shrinks its weight. H2's physics part is plain least
squares and is not exposed to that choice.

In [ ]:
ml = runs[runs.experiment.isin(["ML_FEAT", "ML_PHYS"])][["experiment", "fold_id", "alpha", "n_nonzero", "phys_nonzero", "best_val_mae_m", "test_mae", "test_r2"]].copy()
ml[["best_val_mae_m", "test_mae"]] *= 100
ml.rename(columns={"best_val_mae_m": "val_mae_cm", "test_mae": "test_mae_cm"}).round(3)

### Where H2 differs from physics: take-off (first 20%) vs rest of flight

In [ ]:
seg = []
for fold in FOLDS:
    for name in ["PHYS", "E4", "E4_seed43", "E4_seed44"]:
        q = pd.read_csv(ROOT / f"results/predictions/{name}_{fold}.csv")
        e = 100 * (q.y_pred - q.y_true).abs(); n = int(np.ceil(0.2 * len(q)))
        seg.append(dict(fold=fold, run=name, takeoff_mae_cm=e[:n].mean(), rest_mae_cm=e[n:].mean()))
pd.DataFrame(seg).round(2)

## Branch ablation: what each part adds (test MAE, mean over folds, 3 seeds)

Read top to bottom: raw IMU only → + 104 features + 4 physics as inputs (H1) → physics as the output backbone (H2).
`_min10` rows are a sensitivity check added after the first results: the landing-phase validation block
stopped direct models after 1–2 epochs, so these runs may not pick a checkpoint before epoch 10
(chosen from where the *training* curves flatten). Originals are kept.

In [ ]:
steps = [("IMUlstm", "raw IMU, LSTM"), ("IMUlstm_min10", "raw IMU, LSTM (warm-up)"),
         ("IMUcnn", "raw IMU, 1D-CNN"), ("IMUcnn_min10", "raw IMU, 1D-CNN (warm-up)"),
         ("E3", "H1: IMU + features + physics"), ("E3_min10", "H1 (warm-up)"),
         ("E6", "H3: IMU + camera"), ("E6_min10", "H3 (warm-up)"),
         ("E4", "H2: physics + correction (LSTM)"), ("E4cnn", "H2: physics + correction (CNN)"),
         ("PHYS", "physics alone")]
per = summary.set_index(["experiment", "fold_id"])
rows = []
for exp, label in steps:
    if (exp, "mean_of_folds") not in per.index:
        continue
    r = {"experiment": exp, "model": label}
    for f in FOLDS:
        r[f"mae_cm {f[6:11]}"] = 100 * per.loc[(exp, f), "test_mae_mean"]
    m = per.loc[(exp, "mean_of_folds")]
    r.update(mae_cm=100 * m.test_mae_mean, seed_sd_cm=100 * m.test_mae_std, r2=m.test_r2_mean)
    rows.append(r)
ablation = pd.DataFrame(rows).round(3)
ablation.to_csv(ROOT / "results/metrics/hybrid_ablation.csv", index=False)
ablation

## 4. What each part contributes (fold means, test MAE cm)

Read with the seed spread: a difference smaller than about one seed std is not a real effect.

In [ ]:
m = summary[summary.fold_id == "mean_of_folds"].set_index("experiment")
contrib = pd.DataFrame({"test_mae_cm": 100 * m.test_mae_mean, "seed_sd_cm": 100 * m.test_mae_std,
                        "test_r2": m.test_r2_mean, "val_mae_cm": 100 * m.best_val_mae_m_mean}).round(3)
contrib.loc["Lasso (Phase 1)", "test_mae_cm"] = 100 * phase1.query("model=='lasso' and fold_id=='mean'").mae.iloc[0]
contrib

In [ ]:
vec = runs[runs.experiment == "E4vec"]
if len(vec):
    print("E4vec (4-value head): mean |error vector| on test, cm, per fold:")
    print((100 * vec.groupby("fold_id").test_vec_mae_m.agg(["mean", "std"])).round(3))
    for fold in FOLDS:
        v = pd.read_csv(ROOT / f"results/predictions/E4vec_{fold}.csv")
        ph = pd.read_csv(ROOT / f"data/processed/physics_dl/physics_{fold}.csv")
        m = v.merge(ph, on=["seq_id", "window_id"], how="left")
        true = m[["y_true_dx", "y_true_dy", "y_true_dz"]].to_numpy()
        print(fold, "| no correction:", round(100 * np.linalg.norm(true, axis=1).mean(), 2),
              "cm | raw physics vector:", round(100 * np.linalg.norm(true - m[["phys_dx", "phys_dy", "phys_dz"]].to_numpy(), axis=1).mean(), 2),
              "cm | E4vec (seed 42):", round(100 * np.linalg.norm(true - m[["y_pred_dx", "y_pred_dy", "y_pred_dz"]].to_numpy(), axis=1).mean(), 2), "cm")
e7 = runs[runs.experiment == "E7"]
if len(e7):
    print("E7 base chosen on validation:", e7.e7_base.iloc[0])